# MiningMedicinalTaxa — Example Notebook
Extract plant names, medical conditions, and medicinal effects from text using **SciBERT** (NER + RE) and **GPT** (structured extraction).

Both methods return the same `TaxaData` schema so results are directly comparable.

In [ ]:
# Install
# Run once, then RESTART RUNTIME
!pip install -q git+https://github.com/alrichardbollans/wcvpy.git
!pip install -q git+https://github.com/alrichardbollans/MiningMedicinalTaxa.git

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 135.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you ha

In [1]:
 Get the sample text file, or upload your own.
import os, urllib.request

# Clear any previously-uploaded .txt files so we don't accidentally process old input
for f in [f for f in os.listdir('.') if f.endswith('.txt')]:
    os.remove(f)
    print(f'Removed previous upload: {f}')

SAMPLE_URL = "https://raw.githubusercontent.com/alrichardbollans/MiningMedicinalTaxa/main/R/sample.txt"
txt_name = "sample.txt"
urllib.request.urlretrieve(SAMPLE_URL, txt_name)

# To use your own file instead, comment out the line above and uncomment below:
# from google.colab import files
# uploaded = files.upload()
# txt_name = list(uploaded.keys())[0]

print(f"Using: {txt_name}")

KeyboardInterrupt: 

In [ ]:
# set API key (for GPT extraction only)

# Load OpenAI key from Colab secrets (add your apy key clicking the icon in left sidebar)
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
# Pretty-print helper. It works for both SciBERT and GPT output
def print_taxa(taxa_data, title='Results'):
    taxa = taxa_data.taxa or []
    print(f'\n{"=" * 60}')
    print(f'  {title} — {len(taxa)} taxa found')
    print(f'{"=" * 60}')
    for i, taxon in enumerate(taxa, 1):
        print(f'\n  [{i}] {taxon.scientific_name}')
        conditions = taxon.medical_conditions
        effects = taxon.medicinal_effects
        if conditions:
            print(f'      Conditions: {", ".join(str(c) for c in conditions) if isinstance(conditions, list) else conditions}')
        else:
            print(f'      Conditions: —')
        if effects:
            print(f'      Effects:    {", ".join(str(e) for e in effects) if isinstance(effects, list) else effects}')
        else:
            print(f'      Effects:    —')
    print(f'\n{"=" * 60}\n')

## 1. SciBERT Extraction
Fine-tuned SciBERT NER and RE models. Runs locally, no API key needed.

In [ ]:
# Load SciBert models
from SciBert.running_scibert import load_scibert
models = load_scibert()

Root directory: /usr/local/lib/python3.12/dist-packages/SciBert


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored 

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [ ]:
# Extraction
# run_re=True also extracts relations (slower)

from SciBert.running_scibert import query_scibert
from LLM_models.evaluating import clean_model_annotations_using_taxonomy_knowledge
import json

scibert_output = query_scibert(models, txt_name, json_dump='scibert_output.json', run_re=True)
# optional: use a taxonomy filter to remove vernacular names
scibert_output = clean_model_annotations_using_taxonomy_knowledge(scibert_output)

print_taxa(scibert_output, 'SciBERT')


  SciBERT — 56 taxa found

  [1] stangeria
      Conditions: —
      Effects:    —

  [2] lepidozamia
      Conditions: —
      Effects:    —

  [3] encephalartos
      Conditions: —
      Effects:    —

  [4] artocarpus communis
      Conditions: —
      Effects:    —

  [5] cycas circinalis
      Conditions: tropical sores, ulcers, parkinsonism, amyotrophic, wounds, snake bites, sclerosis
      Effects:    —

  [6] encephalartos tensus willd
      Conditions: —
      Effects:    —

  [7] nux) vomica s. f
      Conditions: —
      Effects:    —

  [8] macrozamia peroffskyana
      Conditions: —
      Effects:    —

  [9] macrozamia miquelii f.v.m
      Conditions: —
      Effects:    —

  [10] macrozamia miquelii
      Conditions: —
      Effects:    —

  [11] zamia integrifolia
      Conditions: —
      Effects:    —

  [12] zamia
      Conditions: —
      Effects:    —

  [13] cycas circinalis l
      Conditions: hepatic
      Effects:    —

  [14] macrozamia
      Conditions: —
  

## 2. GPT Extraction


In [ ]:
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from LLM_models.running_models import query_a_model, get_input_size_limit
from LLM_models.evaluating import clean_model_annotations_using_taxonomy_knowledge

# Specify a .env file containing your API key in the form OPENAI_API_KEY="key",
# or alternatively specify the apikey directly with the apikey parameter
load_dotenv(dotenv_path='.env')

# Set up a GPT model
gpt_model = ChatOpenAI(model="gpt-4o-2024-08-06", temperature=0)
context_window = get_input_size_limit(5)  # 5k tokens per chunk (max 128k tokens)

gpt_output = query_a_model(gpt_model, txt_name,
                              context_window, json_dump='gpt_output.json', single_chunk=False)

# If you want to clean outputs by removing annotations with unknown scientific names:
gpt_output = clean_model_annotations_using_taxonomy_knowledge(gpt_output)

print_taxa(gpt_output, 'GPT-4o')


  GPT-4o — 40 taxa found

  [1] cycas circinalis
      Conditions: snake bites, ulcers, nephritic pains
      Effects:    narcotic, neurotoxic

  [2] encephalartos tensus
      Conditions: —
      Effects:    —

  [3] macrozamia peroffskyana
      Conditions: —
      Effects:    —

  [4] macrozamia miquelii
      Conditions: rickets
      Effects:    —

  [5] macrozamia miquelii f.v.m
      Conditions: rickets
      Effects:    —

  [6] zamia integrifolia
      Conditions: —
      Effects:    —

  [7] cycas media r. br
      Conditions: —
      Effects:    —

  [8] cycas circinalis l
      Conditions: neoplastic disease, alzheimer-type dementia, parkinsonism dementia, motor neuron disease, parkinsonism, amyotrophic lateral sclerosis
      Effects:    —

  [9] macrozamia spiralis
      Conditions: —
      Effects:    —

  [10] dioon edule l
      Conditions: neuralgia
      Effects:    —

  [11] zamia portoricensis urban
      Conditions: paralysis
      Effects:    —

  [12] cycas rhu

Outputs from this process (the json_dump files) can be manually verified using our reference verifier shiny app, hosted here: __https://huggingface.co/spaces/alrichardbollans/MedicinalTaxonVerifier__

In [ ]:
files.download('scibert_output.json')
files.download('gpt_output.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>